# Paso 3: Construcción del Modelo de Machine Learning

En este notebook se utilizará el conjunto de datos generado durante el proceso de **Ingeniería de Características** (`Code/2_feature_engineering.ipynb`), el cual contiene las variables de entrada y las etiquetas de falla construidas para el problema de Mantenimiento Predictivo.

Como primer paso, se cargará el archivo **Parquet** generado en el notebook anterior y posteriormente se dividirá el conjunto de datos en subconjuntos de entrenamiento y prueba.

A continuación, se entrenarán y evaluarán diferentes modelos de clasificación supervisada con el objetivo de predecir qué componente de una máquina tiene mayor probabilidad de fallar dentro de la ventana de predicción definida. En este laboratorio se construirán dos modelos:

- Árbol de Decisión (*Decision Tree Classifier*).
- Bosque Aleatorio (*Random Forest Classifier*).

Una vez entrenados, ambos modelos serán comparados utilizando métricas de evaluación para seleccionar el de mejor desempeño.

Finalmente, el modelo seleccionado se almacenará para ser utilizado posteriormente durante el proceso de inferencia y despliegue del servicio de predicción en el notebook **`Code/4_operationalization.ipynb`**.


In [6]:
# ==========================================
# Importación de librerías
# ==========================================

# Utilidades
import os
import glob
import time

# Manipulación de datos
import pandas as pd
import numpy as np

# Persistencia del modelo
import joblib

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocesamiento
from sklearn.preprocessing import (
    LabelEncoder,
    OneHotEncoder
)

# División del conjunto de datos
from sklearn.model_selection import train_test_split

# Modelos de Machine Learning
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

# Pipelines
from sklearn.pipeline import Pipeline

# Métricas de evaluación
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ==========================================
# Tiempo de ejecución del notebook
# ==========================================

tic = time.time()

## Carga del conjunto de datos

En el notebook **`Code/2_feature_engineering.ipynb`** se generó el conjunto de datos final mediante el proceso de Ingeniería de Características y la construcción de las etiquetas de falla.

En esta etapa, dicho conjunto de datos será cargado desde el archivo **Parquet** generado previamente para utilizarlo en el entrenamiento y evaluación de los modelos de Machine Learning.

In [7]:
# ==========================================
# Configuración de rutas
# ==========================================

# Archivo generado en el Notebook 2
INPUT_PATH = "../data/output_data/featureengineering_files.parquet"

# Ruta donde se almacenará el modelo entrenado
MODEL_PATH = "../models/random_forest_model.pkl"

In [8]:
# Cargar el conjunto de datos generado en el Notebook 2
labeled_features = pd.read_parquet(INPUT_PATH)

# Visualizar las primeras filas
labeled_features.head(10)

,machineID,dt_truncated,datetime,volt_rollingmean_12,volt_rollingstd_12,rotate_rollingmean_12,rotate_rollingstd_12,pressure_rollingmean_12,pressure_rollingstd_12,vibration_rollingmean_12,...,comp2sum,comp3sum,comp4sum,age,model_model1,model_model2,model_model3,model_model4,failure,label_e
0,1,2015-01-01 00:00:00,2015-01-01 08:30:00,169.512597,6.239068,425.853115,46.581966,101.886267,13.166950,41.119335,...,213.0,153.0,168.0,18,0,0,1,0,0.0,0.0
1,1,2015-01-01 12:00:00,2015-01-01 17:30:00,167.007032,8.260186,434.948972,50.000568,100.970047,10.863588,39.412711,...,214.0,154.0,169.0,18,0,0,1,0,0.0,4.0
2,1,2015-01-02 00:00:00,2015-01-02 05:30:00,171.754734,13.479925,454.534311,44.925868,93.989345,7.963589,40.265190,...,214.0,154.0,169.0,18,0,0,1,0,0.0,4.0
3,1,2015-01-02 12:00:00,2015-01-02 17:30:00,168.184995,15.356860,444.596343,39.260749,100.778017,10.107193,39.209928,...,215.0,155.0,170.0,18,0,0,1,0,0.0,4.0
4,1,2015-01-03 00:00:00,2015-01-03 05:30:00,172.717240,12.879603,468.079805,40.727721,99.586915,12.005493,40.406556,...,215.0,155.0,170.0,18,0,0,1,0,0.0,4.0
5,1,2015-01-03 12:00:00,2015-01-03 17:30:00,170.834194,12.090919,457.057988,41.703938,96.743024,9.012532,50.676883,...,216.0,156.0,171.0,18,0,0,1,0,0.0,4.0
6,1,2015-01-04 00:00:00,2015-01-04 05:30:00,172.869308,14.466782,442.555398,42.292101,103.639622,9.550445,52.490685,...,216.0,156.0,171.0,18,0,0,1,0,0.0,4.0
7,1,2015-01-04 12:00:00,2015-01-04 17:30:00,172.778379,21.972799,446.558222,34.547407,100.077137,10.305353,51.194922,...,217.0,157.0,172.0,18,0,0,1,0,0.0,4.0
8,1,2015-01-05 00:00:00,2015-01-05 05:30:00,170.376832,19.677497,439.599809,39.904096,97.821329,8.926671,50.729673,...,217.0,157.0,172.0,18,0,0,1,0,4.0,4.0
9,1,2015-01-05 12:00:00,2015-01-05 17:30:00,174.080665,14.994161,456.036474,49.178578,104.582541,11.473903,41.209276,...,218.0,158.0,173.0,18,0,0,1,0,0.0,0.0


## Preparación de los datos de entrenamiento y prueba

Antes de entrenar un modelo de Machine Learning, es necesario dividir el conjunto de datos en dos subconjuntos:

- **Conjunto de entrenamiento (Training Set):** utilizado para que el algoritmo aprenda los patrones presentes en los datos.
- **Conjunto de prueba (Test Set):** utilizado para evaluar el desempeño del modelo sobre datos que no fueron utilizados durante el entrenamiento.

Esta separación permite obtener una estimación más realista de la capacidad del modelo para realizar predicciones sobre nuevos datos y ayuda a identificar posibles problemas de sobreajuste (*overfitting*).

En este laboratorio se utilizará una división del **70 %** de los datos para entrenamiento y el **30 %** para prueba, una proporción ampliamente utilizada en problemas de clasificación.

En las siguientes celdas se separarán las variables predictoras (**X**) de la variable objetivo (**y**) y posteriormente se realizará la partición del conjunto de datos para entrenar y evaluar los modelos de clasificación.


In [13]:
# ==========================================
# Definición de variables para el modelado
# ==========================================

# Variable objetivo
label_var = "label_e"

# Columnas que no se utilizarán como variables predictoras
key_cols = ["machineID", "dt_truncated"]

remove_names = key_cols + [
    "datetime",
    "failure",
    "model",
    "label_e"
]

# Variables predictoras
input_features = [
    col for col in labeled_features.columns
    if col not in remove_names
]

# Visualizar las variables de entrada
input_features

['volt_rollingmean_12',
 'volt_rollingstd_12',
 'rotate_rollingmean_12',
 'rotate_rollingstd_12',
 'pressure_rollingmean_12',
 'pressure_rollingstd_12',
 'vibration_rollingmean_12',
 'vibration_rollingstd_12',
 'volt_rollingmean_24',
 'volt_rollingstd_24',
 'rotate_rollingmean_24',
 'rotate_rollingstd_24',
 'pressure_rollingmean_24',
 'pressure_rollingstd_24',
 'vibration_rollingmean_24',
 'vibration_rollingstd_24',
 'volt_rollingmean_36',
 'volt_rollingstd_36',
 'rotate_rollingmean_36',
 'rotate_rollingstd_36',
 'pressure_rollingmean_36',
 'pressure_rollingstd_36',
 'vibration_rollingmean_36',
 'vibration_rollingstd_36',
 'error1sum_rollingmean_24',
 'error2sum_rollingmean_24',
 'error3sum_rollingmean_24',
 'error4sum_rollingmean_24',
 'error5sum_rollingmean_24',
 'comp1sum',
 'comp2sum',
 'comp3sum',
 'comp4sum',
 'age',
 'model_model1',
 'model_model2',
 'model_model3',
 'model_model4']

Una vez definidas las variables predictoras, se construirá la matriz de características (**X**) y la variable objetivo (**y**). Posteriormente, el conjunto de datos se dividirá en datos de entrenamiento y prueba utilizando la función `train_test_split` de Scikit-learn.

El conjunto de entrenamiento se utilizará para ajustar los modelos de Machine Learning, mientras que el conjunto de prueba permitirá evaluar su capacidad para realizar predicciones sobre datos no vistos durante el entrenamiento. En este laboratorio se empleará una división del **70 %** para entrenamiento y **30 %** para prueba.

In [14]:
# ==========================================
# Preparación de los datos para el entrenamiento
# ==========================================

# Variables predictoras
X = labeled_features[input_features]

# Variable objetivo
y = labeled_features[label_var]

# División en entrenamiento (70%) y prueba (30%)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Mostrar el tamaño de cada conjunto
print(f"Registros de entrenamiento: {X_train.shape[0]}")
print(f"Registros de prueba: {X_test.shape[0]}")

Registros de entrenamiento: 51200
Registros de prueba: 21943


# Modelos de clasificación

En los problemas de Mantenimiento Predictivo es común que las fallas representen una pequeña proporción del total de las observaciones, ya que la mayoría de las máquinas operan correctamente durante la mayor parte del tiempo. Esta situación genera un **desbalance de clases**, donde la clase correspondiente al funcionamiento normal contiene muchos más registros que las clases asociadas a las fallas.

Cuando existe este desbalance, un modelo puede alcanzar una alta exactitud (*Accuracy*) simplemente prediciendo la clase mayoritaria, pero presentar un bajo desempeño al identificar las fallas, que son precisamente los eventos de mayor interés.

Por esta razón, además de la exactitud, es importante evaluar el modelo utilizando métricas como la **precisión (Precision)**, la **sensibilidad (Recall)** y el **F1-Score**, las cuales permiten medir de forma más adecuada el desempeño del clasificador en cada una de las clases.

En este laboratorio se entrenarán y compararán dos modelos de clasificación ampliamente utilizados:

- **Árbol de Decisión (Decision Tree Classifier):** modelo basado en reglas de decisión que divide los datos de manera jerárquica para realizar la clasificación. Es fácil de interpretar, admite problemas multiclase y permite capturar relaciones no lineales entre las variables.

- **Bosque Aleatorio (Random Forest Classifier):** modelo de ensamble conformado por múltiples árboles de decisión. Cada árbol se entrena sobre diferentes subconjuntos de datos y variables, y la predicción final se obtiene mediante votación. Esta estrategia suele ofrecer una mayor capacidad de generalización y reduce el riesgo de sobreajuste (*Overfitting*).

En las siguientes secciones se entrenarán ambos modelos utilizando el conjunto de entrenamiento y posteriormente se comparará su desempeño sobre el conjunto de prueba para seleccionar el modelo con mejores resultados.


In [15]:
# ==========================================
# Selección y entrenamiento del modelo
# ==========================================

# Seleccionar el modelo: "DecisionTree" o "RandomForest"
model_type = "RandomForest"

if model_type == "DecisionTree":

    model = DecisionTreeClassifier(
        criterion="gini",      # Criterio de división
        max_depth=15,          # Profundidad máxima del árbol
        min_samples_split=2,   # Mínimo de muestras para dividir un nodo
        min_samples_leaf=1,    # Mínimo de muestras por hoja
        random_state=42
    )

else:

    model = RandomForestClassifier(
        n_estimators=50,       # Número de árboles
        criterion="gini",      # Criterio de división
        max_depth=15,          # Profundidad máxima de cada árbol
        max_features="sqrt",   # Variables evaluadas en cada división
        bootstrap=True,        # Muestreo con reemplazo
        random_state=42,
        n_jobs=-1              # Utilizar todos los núcleos disponibles
    )

# Entrenar el modelo
model.fit(X_train, y_train)

print(f"Modelo entrenado: {model_type}")

Modelo entrenado: RandomForest


In [16]:
X_train.dtypes

volt_rollingmean_12         float64
volt_rollingstd_12          float64
rotate_rollingmean_12       float64
rotate_rollingstd_12        float64
pressure_rollingmean_12     float64
pressure_rollingstd_12      float64
vibration_rollingmean_12    float64
vibration_rollingstd_12     float64
volt_rollingmean_24         float64
volt_rollingstd_24          float64
rotate_rollingmean_24       float64
rotate_rollingstd_24        float64
pressure_rollingmean_24     float64
pressure_rollingstd_24      float64
vibration_rollingmean_24    float64
vibration_rollingstd_24     float64
volt_rollingmean_36         float64
volt_rollingstd_36          float64
rotate_rollingmean_36       float64
rotate_rollingstd_36        float64
pressure_rollingmean_36     float64
pressure_rollingstd_36      float64
vibration_rollingmean_36    float64
vibration_rollingstd_36     float64
error1sum_rollingmean_24    float64
error2sum_rollingmean_24    float64
error3sum_rollingmean_24    float64
error4sum_rollingmean_24    

## Evaluación del modelo

Una vez entrenado el modelo, el siguiente paso consiste en evaluar su desempeño utilizando el conjunto de datos de prueba.

Para ello, el modelo realizará predicciones sobre observaciones que no fueron utilizadas durante el proceso de entrenamiento. Esto permite medir su capacidad de generalización y estimar cómo se comportaría al realizar predicciones sobre nuevos datos.

Posteriormente, las predicciones obtenidas se compararán con las etiquetas reales para calcular diferentes métricas de evaluación, como la **Exactitud (Accuracy)**, **Precisión (Precision)**, **Sensibilidad (Recall)** y **F1-Score**. Estas métricas permitirán comparar el desempeño de los diferentes modelos de clasificación y seleccionar la alternativa con mejores resultados para su posterior utilización en la etapa de inferencia.

In [17]:
# ==========================================
# Predicciones sobre el conjunto de prueba
# ==========================================

# Realizar las predicciones
y_pred = model.predict(X_test)

# Construir la matriz de confusión
conf_matrix = confusion_matrix(y_test, y_pred)

# Mostrar la matriz de confusión como un DataFrame
confuse = pd.DataFrame(
    conf_matrix,
    index=np.unique(y_test),
    columns=np.unique(y_test)
)

confuse

,0.0,1.0,2.0,3.0,4.0
0.0,20222,8,3,1,2
1.0,268,132,0,3,1
2.0,350,0,202,1,7
3.0,191,1,0,106,2
4.0,268,5,6,1,163


## Interpretación de la matriz de confusión y métricas de evaluación

La **matriz de confusión** permite comparar las clases reales con las clases predichas por el modelo. En este caso, las filas representan las etiquetas reales y las columnas las predicciones realizadas por el clasificador.

Cada valor de la diagonal principal corresponde al número de observaciones clasificadas correctamente. Por el contrario, los valores fuera de la diagonal representan errores de clasificación, es decir, casos en los que el modelo asignó una clase diferente a la real.

Aunque la matriz de confusión proporciona una visión detallada del desempeño del modelo, es habitual resumir estos resultados mediante diferentes métricas de evaluación.

En este laboratorio se calcularán las siguientes métricas:

- **Exactitud (Accuracy):** indica la proporción total de observaciones clasificadas correctamente. Aunque es una métrica ampliamente utilizada, puede resultar engañosa cuando existe un desbalance entre las clases.

- **Precisión (Precision):** mide la proporción de predicciones positivas que realmente corresponden a la clase correcta. Un valor alto de precisión indica que el modelo genera pocos falsos positivos.

- **Sensibilidad (Recall):** mide la capacidad del modelo para identificar correctamente las observaciones pertenecientes a una determinada clase. Un valor alto de sensibilidad implica que el modelo produce pocos falsos negativos.

- **F1-Score:** combina la precisión y la sensibilidad en una única métrica mediante su media armónica. Es especialmente útil cuando existe un desbalance entre las clases, ya que ofrece una evaluación más equilibrada del desempeño del modelo.

En las siguientes celdas se calcularán estas métricas para evaluar el desempeño del modelo de clasificación y posteriormente comparar los resultados obtenidos entre los diferentes algoritmos implementados en este laboratorio.


In [18]:
# ==========================================
# Evaluación del modelo
# ==========================================

# Calcular las métricas de evaluación
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted")
recall = recall_score(y_test, y_pred, average="weighted")
f1 = f1_score(y_test, y_pred, average="weighted")

# Mostrar los resultados
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-Score : {f1:.4f}")

# Mostrar el reporte de clasificación
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred))

Accuracy : 0.9490
Precision: 0.9484
Recall   : 0.9490
F1-Score : 0.9379

Reporte de clasificación:
              precision    recall  f1-score   support

         0.0       0.95      1.00      0.97     20236
         1.0       0.90      0.33      0.48       404
         2.0       0.96      0.36      0.52       560
         3.0       0.95      0.35      0.51       300
         4.0       0.93      0.37      0.53       443

    accuracy                           0.95     21943
   macro avg       0.94      0.48      0.60     21943
weighted avg       0.95      0.95      0.94     21943



In [19]:
# ==========================================
# Importancia de las variables
# ==========================================

# Obtener la importancia de las variables
feature_importance = pd.DataFrame({
    "Variable": input_features,
    "Importancia": model.feature_importances_
})

# Ordenar de mayor a menor importancia
feature_importance = feature_importance.sort_values(
    by="Importancia",
    ascending=False
)

# Visualizar las variables más importantes
feature_importance.head(20)

,Variable,Importancia
28,error5sum_rollingmean_24,0.087442
2,rotate_rollingmean_12,0.073827
24,error1sum_rollingmean_24,0.071135
26,error3sum_rollingmean_24,0.065700
25,error2sum_rollingmean_24,0.050650
10,rotate_rollingmean_24,0.044991
0,volt_rollingmean_12,0.043708
27,error4sum_rollingmean_24,0.043631
8,volt_rollingmean_24,0.039957
6,vibration_rollingmean_12,0.038788


## Interpretación de los resultados

Los resultados obtenidos en este laboratorio corresponden a un conjunto de datos utilizado con fines educativos. En un escenario real de Mantenimiento Predictivo, el comportamiento del modelo puede variar debido a factores como la calidad de los datos, el desbalance entre las clases, la presencia de ruido y la complejidad de las condiciones de operación de los equipos.

Por esta razón, además de analizar la **Exactitud (Accuracy)**, es recomendable evaluar métricas como la **Precisión (Precision)**, la **Sensibilidad (Recall)** y el **F1-Score**, ya que ofrecen una visión más completa del desempeño del modelo, especialmente cuando el objetivo es identificar eventos de falla poco frecuentes.

Una vez evaluados los modelos, se seleccionará el de mejor desempeño para utilizarlo en la etapa de inferencia.

## Almacenamiento del modelo

Como paso final, el modelo entrenado se almacenará en un archivo para reutilizarlo posteriormente sin necesidad de volver a entrenarlo.

Este modelo será utilizado en el siguiente notebook (**`Code/4_operationalization.ipynb`**), donde se cargará para realizar predicciones sobre nuevos datos y simular el proceso de inferencia de una solución de Mantenimiento Predictivo.


In [20]:
# ==========================================
# Guardar el modelo entrenado
# ==========================================

# Guardar el modelo en un archivo
joblib.dump(model, MODEL_PATH)

print(f"Modelo guardado correctamente en: {MODEL_PATH}")

# Tiempo total de ejecución del notebook
toc = time.time()

print(f"Tiempo total de ejecución: {(toc - tic)/60:.2f} minutos")

Modelo guardado correctamente en: ../models/random_forest_model.pkl
Tiempo total de ejecución: 3.02 minutos


## Conclusión

En este notebook se cargó el conjunto de datos generado durante la etapa de Ingeniería de Características, se prepararon los datos para el entrenamiento y se construyeron dos modelos de clasificación: un **Árbol de Decisión (Decision Tree)** y un **Bosque Aleatorio (Random Forest)**.

Posteriormente, ambos modelos fueron evaluados mediante diferentes métricas de desempeño y se almacenó el modelo seleccionado para su utilización en la siguiente etapa del laboratorio.

En el próximo notebook, **`Code/4_operationalization.ipynb`**, se cargará el modelo entrenado para realizar predicciones sobre nuevos datos, simulando el proceso de inferencia de una solución de Mantenimiento Predictivo.